## USE CASE OF PANDAS DATA ANALYISIS 

You are a data engineer at the airport of noakchott and the management would like to analyse data about flights in order to answer the folowing questions 


Data acquisition

1 - Go to https://rapidapi.com/aedbx-aedbx/api/aerodatabox/playground/apiendpoint_97755564-247f-411f-bef2-ad26a453e389 <br/>
2 - Create an account(free plan)<br/>
3- Write api call to get airpot data and save it into './data/airport.json' request from (2025-01-01 untill '2025-02-02'<br/>
4- import data  <br/>


## Data analyis <br/>


Q1. How many flights does the dataset contain in total? What is the minimum data and maximum data that we have <br/>
Q2. How many distinct airlines are represented?<br/>
Q3. Fleet mix: Which aircraft models are used most and how many times?<br/>
Q4. Busiest arrival airport: List the top‑5 arrival airports by number of scheduled arrivals.<br/>
Q5. Schedule accuracy: For flights where arrival.revisedTime is available, calculate the average arrival delay (minutes).<br/>
Q6. Route mapping: Create a mini‑table with the top‑3 most common city‑pairs (origin‑>destination IATA codes) and the associated flight counts.<br/>

# GOOD LUCK

# By Ahmed Abd Dayme Ahmed Bouha 

In [46]:
print("Hello bro ,it's me Ahmed Abd Dayme Ahmed Bouha")

Hello bro ,it's me Ahmed Abd Dayme Ahmed Bouha


## Data acquisition

In [55]:
import requests
import json
from datetime import datetime
import pandas as pd
import os

# Create data directory if it doesn't exist
os.makedirs('./data', exist_ok=True)

# API configuration
url = "https://aerodatabox.p.rapidapi.com/flights/airports/icao/GQNN"

headers = {
    "X-RapidAPI-Key": "5eeb50881amsh54f541f9283f2c5p1e19d9jsn554ac6d9e699", 
    "X-RapidAPI-Host": "aerodatabox.p.rapidapi.com"
}

querystring = {
    "withLeg": "true",
    "withCancelled": "true",
    "withCodeshared": "true",
    "withCargo": "true",
    "withPrivate": "true",
    "withLocation": "true",
    "from": "2025-01-01",
    "to": "2025-02-02"
}

try:
    # Make the API request with SSL verification disabled if needed
    response = requests.get(url, headers=headers, params=querystring, verify=False)
    
    # Check response
    if response.status_code == 200:
        with open('./data/airport.json', 'w') as f:
            json.dump(response.json(), f)
        print("Success! Data saved to airport.json")
    else:
        print(f"Error: API returned status code {response.status_code}")
        print(f"Error message: {response.text}")
        
except requests.exceptions.RequestException as e:
    print(f"Error making the request: {str(e)}")

/home/ahmed/Desktop/rimai/rimai/courses/data-engineering/course-2/.venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aerodatabox.p.rapidapi.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Success! Data saved to airport.json


## Import data and start Data analysis

In [56]:
import json
import pandas as pd
from datetime import datetime

# Read the JSON data
with open('./data/airport.json', 'r') as f:
    data = json.load(f)

# Process the data
flights = []
for flight_type in ['arrivals', 'departures']:
    if flight_type in data:
        for flight in data[flight_type]:
            scheduled_time = (flight.get('arrival', {}).get('scheduledTime', {}).get('local', '') 
                            if flight_type == 'arrivals' 
                            else flight.get('departure', {}).get('scheduledTime', {}).get('local', ''))
            
            # Get revised time for delay calculation
            revised_time = (flight.get('arrival', {}).get('revisedTime', {}).get('local', '')
                          if flight_type == 'arrivals' else '')
            
            flight_info = {
                'type': flight_type,
                'flight_number': flight.get('number', ''),
                'airline': flight.get('airline', {}).get('name', ''),
                'aircraft': flight.get('aircraft', {}).get('model', ''),
                'origin': flight.get('departure', {}).get('airport', {}).get('name', ''),
                'destination': flight.get('arrival', {}).get('airport', {}).get('name', ''),
                'origin_iata': flight.get('departure', {}).get('airport', {}).get('iata', ''),
                'destination_iata': flight.get('arrival', {}).get('airport', {}).get('iata', ''),
                'scheduled_time': scheduled_time,
                'revised_time': revised_time,
                'status': flight.get('status', '')
            }
            flights.append(flight_info)

df = pd.DataFrame(flights)

print("Analysis Results:")
print("-" * 50)


Analysis Results:
--------------------------------------------------


In [57]:
# Q1: How many flights does the dataset contain in total? What is the minimum data and maximum data that we have
print("\nQ1: Total flights and date range")
print(f"Total flights: {len(df)}")
if not df.empty:
    print(f"Time period: {df['scheduled_time'].min()} to {df['scheduled_time'].max()}")



Q1: Total flights and date range
Total flights: 10
Time period: 2025-04-27 16:10+00:00 to 2025-04-28 02:40+00:00


In [58]:
# Q2: How many distinct airlines are represented?
print("\nQ2: Distinct airlines")
airlines = df['airline'].dropna().unique()
print(f"Number of distinct airlines: {len(airlines)}")
print("Airlines:", ', '.join([str(a) for a in airlines if str(a).strip()]))



Q2: Distinct airlines
Number of distinct airlines: 5
Airlines: Turkish, Air Senegal, Air France, Mauritania  International, Royal Air Maroc


In [59]:
# Q3: Fleet mix: Which aircraft models are used most and how many times?
print("\nQ3: Aircraft models usage")
aircraft_counts = df['aircraft'].value_counts()
print(aircraft_counts)


Q3: Aircraft models usage
aircraft
Airbus A330-200    2
ATR 72             2
Boeing 787-9       2
Embraer 175        2
Boeing 737-800     2
Name: count, dtype: int64


In [60]:
# Q4: Busiest arrival airport: List the top‑5 arrival airports by number of scheduled arrivals
print("\nQ4: Top 5 arrival airports")
arrivals = df[df['type'] == 'arrivals']['destination'].value_counts().head(5)
print(arrivals)


Q4: Top 5 arrival airports
destination
    5
Name: count, dtype: int64


In [61]:
# Q5: Schedule accuracy: For flights where arrival.revisedTime is available, calculate the average arrival delay
print("\nQ5: Average arrival delay")
def calculate_delay(row):
    if pd.notna(row['revised_time']) and pd.notna(row['scheduled_time']):
        try:
            revised = pd.to_datetime(row['revised_time'])
            scheduled = pd.to_datetime(row['scheduled_time'])
            return (revised - scheduled).total_seconds() / 60
        except:
            return None
    return None

df['delay_minutes'] = df.apply(calculate_delay, axis=1)
avg_delay = df['delay_minutes'].mean()
print(f"Average delay: {avg_delay if pd.notna(avg_delay) else 'No delay data available'} minutes")



Q5: Average arrival delay
Average delay: No delay data available minutes


In [62]:

# Q6: Route mapping: Create a mini‑table with the top‑3 most common city‑pairs (origin‑>destination) and associated flight counts
print("\nQ6: Top 3 most common city pairs")
df['city_pair'] = df.apply(lambda x: f"{x['origin_iata']} -> {x['destination_iata']}" 
                          if pd.notna(x['origin_iata']) and pd.notna(x['destination_iata']) 
                          else None, axis=1)
top_routes = df['city_pair'].value_counts().head(3)
print("\nTop 3 most common routes (IATA codes):")
print(top_routes)

# Show the same with city names for clarity
df['city_pair_names'] = df.apply(lambda x: f"{x['origin']} -> {x['destination']}" 
                                if pd.notna(x['origin']) and pd.notna(x['destination']) 
                                else None, axis=1)
print("\nTop 3 most common routes (City names):")
print(df['city_pair_names'].value_counts().head(3))


Q6: Top 3 most common city pairs

Top 3 most common routes (IATA codes):
city_pair
CMN ->     2
 -> CMN    2
DSS ->     1
Name: count, dtype: int64

Top 3 most common routes (City names):
city_pair_names
Casablanca ->     2
 -> Casablanca    2
Dakar ->          1
Name: count, dtype: int64


## DONE :-)